In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import joblib
import json


from pathlib import Path
from importlib.metadata import version

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)


RANDOM_STATE = 42
MAX_FLAGGED_RATE = 0.30
pd.set_option("display.width", 120)

In [2]:

# Adjust path according to notebook location
DATA_PATH = Path("../data/loan_applications_cleaned_dataset.csv")

df = pd.read_csv(DATA_PATH)

TARGET = "defaulted"

# Ensure target is binary
assert df[TARGET].notna().all()
assert df[TARGET].isin([0, 1]).all()

# Separate features and target
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(y.value_counts(normalize=True))


Dataset shape: (25000, 11)

Target distribution:
defaulted
0    0.79876
1    0.20124
Name: proportion, dtype: float64


In [3]:

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Development:", X_dev.shape)
print("Final test:", X_test.shape)

print("\nDevelopment target distribution:")
print(y_dev.value_counts(normalize=True))

print("\nFinal test target distribution:")
print(y_test.value_counts(normalize=True))


Development: (20000, 10)
Final test: (5000, 10)

Development target distribution:
defaulted
0    0.79875
1    0.20125
Name: proportion, dtype: float64

Final test target distribution:
defaulted
0    0.7988
1    0.2012
Name: proportion, dtype: float64


In [4]:

X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    stratify=y_dev,
    random_state=RANDOM_STATE
)

print("Fit:", X_train.shape)
print("Validation:", X_val.shape)


Fit: (15000, 10)
Validation: (5000, 10)


In [5]:

numeric_features = X.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), ## filling missing value on columns annual_income and credit_score
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), ## filling missing value on categorical columns
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


Numeric features: ['applicant_age', 'annual_income', 'credit_score', 'loan_amount', 'loan_term_months', 'interest_rate', 'debt_to_income_pct', 'previous_defaults']
Categorical features: ['employment_status', 'home_ownership']


In [6]:
##Creating majority-class baseline

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_val_pred = baseline.predict(X_val)

print("Majority-class baseline (validation)")
print("Accuracy:", accuracy_score(y_val, baseline_val_pred))
print("Precision:", precision_score(
    y_val, baseline_val_pred, zero_division=0
))
print("Recall:", recall_score(
    y_val, baseline_val_pred, zero_division=0
))


Majority-class baseline (validation)
Accuracy: 0.7988
Precision: 0.0
Recall: 0.0


In [7]:
# Creating model pipeline for Logistic Regression
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

# Probability of loan default (class 1). Used for threshold-based classification.
val_proba = model.predict_proba(X_val)[:, 1]

print("Logistic Regression fitted successfully.")


Logistic Regression fitted successfully.


## Validation threshold sweep and 30% manual-review policy

Threshold selection uses **validation data only**. We evaluate each distinct probability boundary, then select the smallest feasible threshold (maximizing review coverage and recall within the 30% capacity). Tied scores are handled together. The final-test data are not used for this decision.

In [8]:
# Validation-only threshold sweep over all distinct score boundaries.
# A threshold strictly above a score excludes all applications tied at that score.
def evaluate_threshold(y_true, probabilities, threshold):
    predicted = (np.asarray(probabilities) >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "flagged_count": int(predicted.sum()),
        "flagged_rate": float(predicted.mean()),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "recall": float(recall_score(y_true, predicted, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, predicted)),
    }

candidate_thresholds = np.unique(np.concatenate([
    np.asarray(val_proba),
    np.nextafter(np.asarray(val_proba), np.inf),
    [0.5, 1.0]
]))
candidate_thresholds = candidate_thresholds[
    np.isfinite(candidate_thresholds) &
    (candidate_thresholds > 0) & (candidate_thresholds < 1)
]

sweep_results = pd.DataFrame([
    evaluate_threshold(y_val, val_proba, threshold)
    for threshold in candidate_thresholds
])
max_flags = int(np.floor(len(y_val) * MAX_FLAGGED_RATE))
sweep_results["within_capacity"] = sweep_results["flagged_count"] <= max_flags
feasible = sweep_results.loc[sweep_results["within_capacity"]]
assert not feasible.empty, "No feasible threshold in (0, 1)."
selected_row = feasible.sort_values(
    ["flagged_count", "threshold"], ascending=[False, True]
).iloc[0]
selected_threshold = float(selected_row["threshold"])
val_pred = (val_proba >= selected_threshold).astype(int)
val_flagged_rate = float(val_pred.mean())
assert val_pred.sum() <= max_flags

print(f"Selected threshold: {selected_threshold:.12g}")
print(f"Validation flagged rate: {val_flagged_rate:.2%} (capacity: {MAX_FLAGGED_RATE:.0%})")
print(f"Validation precision: {precision_score(y_val, val_pred, zero_division=0):.4f}")
print(f"Validation recall: {recall_score(y_val, val_pred, zero_division=0):.4f}")
print(f"Validation accuracy: {accuracy_score(y_val, val_pred):.4f}")
print(f"Candidate thresholds evaluated: {len(sweep_results):,}")

Selected threshold: 0.245233103123
Validation flagged rate: 30.00% (capacity: 30%)
Validation precision: 0.4700
Validation recall: 0.7008
Validation accuracy: 0.7808
Candidate thresholds evaluated: 10,001


In [9]:
# Comparing model performance with majority-class baseline

validation_results = pd.DataFrame([
    {
        "Model": "Majority-class baseline",
        "Flagged Rate": baseline_val_pred.mean(),
        "Precision": precision_score(
            y_val, baseline_val_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, baseline_val_pred, zero_division=0
        ),
        "Accuracy": accuracy_score(
            y_val, baseline_val_pred
        )
    },
    {
        "Model": "Logistic Regression",
        "Flagged Rate": val_pred.mean(),
        "Precision": precision_score(
            y_val, val_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_pred, zero_division=0
        ),
        "Accuracy": accuracy_score(
            y_val, val_pred
        )
    }
])

display(validation_results.round(4))


,Model,Flagged Rate,Precision,Recall,Accuracy
0,Majority-class baseline,0.0,0.00,0.0000,0.7988
1,Logistic Regression,0.3,0.47,0.7008,0.7808


## Validation comparison: selected threshold versus 0.5

The table quantifies the **benefit and cost** of the capacity-constrained threshold relative to the conventional 0.5 decision rule. Differences are in proportions (multiply by 100 for percentage points). This comparison uses validation predictions only.

In [10]:
threshold_05 = evaluate_threshold(y_val, val_proba, 0.5)
threshold_selected = evaluate_threshold(y_val, val_proba, selected_threshold)

comparison = pd.DataFrame([
    {"policy": "Conventional 0.5", **threshold_05},
    {"policy": "Capacity-constrained", **threshold_selected},
]).set_index("policy")
display(comparison.round(4))

tradeoff = pd.DataFrame({
    "at_0.5": pd.Series(threshold_05),
    "selected": pd.Series(threshold_selected),
}).loc[["flagged_rate", "precision", "recall", "accuracy"]]
tradeoff["change_percentage_points"] = (tradeoff["selected"] - tradeoff["at_0.5"]) * 100
display(tradeoff.round(3))

for metric in tradeoff.index:
    change = tradeoff.loc[metric, "change_percentage_points"]
    print(f"{metric}: {change:+.2f} percentage points vs 0.5")
print("Note: precision may fall as recall and review workload increase; read the actual results above.")

,threshold,flagged_count,flagged_rate,precision,recall,accuracy
policy,,,,,,
Conventional 0.5,0.5000,565,0.113,0.6619,0.3718,0.8354
Capacity-constrained,0.2452,1500,0.300,0.4700,0.7008,0.7808


,at_0.5,selected,change_percentage_points
flagged_rate,0.113,0.300,18.700
precision,0.662,0.470,-19.195
recall,0.372,0.701,32.903
accuracy,0.835,0.781,-5.460


flagged_rate: +18.70 percentage points vs 0.5
precision: -19.19 percentage points vs 0.5
recall: +32.90 percentage points vs 0.5
accuracy: -5.46 percentage points vs 0.5
Note: precision may fall as recall and review workload increase; read the actual results above.


## Reproducibility and pipeline configuration

Both splits are stratified and use `RANDOM_STATE = 42`. The pipeline is **numeric median imputation → standard scaling** and **categorical most-frequent imputation → one-hot encoding**, followed by Logistic Regression. The majority-class baseline is fitted on the fit subset.

In [11]:
print("Random seed:", RANDOM_STATE)
print("Development rows:", len(X_dev), "Final-test rows:", len(X_test))
print("Fit rows:", len(X_train), "Validation rows:", len(X_val))
print("Split 1: test_size=0.20, stratify=y")
print("Split 2: test_size=0.25 of development, stratify=y_dev")
print("Pipeline steps:")
for name, step in model.steps:
    print(" -", name, ":", type(step).__name__)
print("Numeric preprocessing:", [name for name, _ in numeric_transformer.steps])
print("Categorical preprocessing:", [name for name, _ in categorical_transformer.steps])
print("Classifier:", model.named_steps["classifier"])
print("Majority baseline:", baseline.strategy)
print("Baseline validation accuracy:", accuracy_score(y_val, baseline_val_pred))

Random seed: 42
Development rows: 20000 Final-test rows: 5000
Fit rows: 15000 Validation rows: 5000
Split 1: test_size=0.20, stratify=y
Split 2: test_size=0.25 of development, stratify=y_dev
Pipeline steps:
 - preprocessor : ColumnTransformer
 - classifier : LogisticRegression
Numeric preprocessing: ['imputer', 'scaler']
Categorical preprocessing: ['imputer', 'encoder']
Classifier: LogisticRegression(max_iter=2000, random_state=42)
Majority baseline: most_frequent
Baseline validation accuracy: 0.7988


In [12]:
# Training on all development data

model.fit(X_dev, y_dev)

print("Model refitted on all development data.")
print("Fixed threshold:", selected_threshold)


Model refitted on all development data.
Fixed threshold: 0.24523310312331778


In [13]:
# Evaluating model final metrics on the test set

test_proba = model.predict_proba(X_test)[:, 1]

# Apply validation-selected threshold
test_pred = (
    test_proba >= selected_threshold
).astype(int)

# Calculate required metrics
test_results = {
    "Flagged Rate": test_pred.mean(),
    "Precision": precision_score(
        y_test, test_pred, zero_division=0
    ),
    "Recall": recall_score(
        y_test, test_pred, zero_division=0
    ),
    "Accuracy": accuracy_score(y_test, test_pred)
}

print("FINAL TEST RESULTS")
print(f"Threshold: {selected_threshold:.6f}")

for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, test_pred, zero_division=0
))

print(
    "\nThe threshold and model configuration were fixed "
    "before final-test evaluation. Nothing was retuned "
    "after seeing these results."
)


FINAL TEST RESULTS
Threshold: 0.245233
Flagged Rate: 0.2848
Precision: 0.4965
Recall: 0.7028
Accuracy: 0.7968

Confusion Matrix:
[[3277  717]
 [ 299  707]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.82      0.87      3994
           1       0.50      0.70      0.58      1006

    accuracy                           0.80      5000
   macro avg       0.71      0.76      0.72      5000
weighted avg       0.83      0.80      0.81      5000


The threshold and model configuration were fixed before final-test evaluation. Nothing was retuned after seeing these results.


In [14]:
## Saving Model and operating threshold

MODEL_VERSION = 1
MODEL_NAME = "loan_default_model_v1"

OUTPUT_DIR = Path("../models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / f"{MODEL_NAME}.joblib"
METADATA_PATH = OUTPUT_DIR / f"{MODEL_NAME}.json"

from datetime import datetime, timezone

feature_names = model.feature_names_in_.tolist()
assert feature_names == X_dev.columns.tolist()
trained_on = datetime.now(timezone.utc).isoformat()


In [15]:
## Collecting Metrics
validation_metrics = {
    "flagged_rate": float(val_pred.mean()),
    "precision": float(
        precision_score(y_val, val_pred, zero_division=0)
    ),
    "recall": float(
        recall_score(y_val, val_pred, zero_division=0)
    ),
    "accuracy": float(
        accuracy_score(y_val, val_pred)
    ),
    "sample_count": int(len(y_val))
}

final_test_metrics = {
    "flagged_rate": float(test_pred.mean()),
    "precision": float(
        precision_score(y_test, test_pred, zero_division=0)
    ),
    "recall": float(
        recall_score(y_test, test_pred, zero_division=0)
    ),
    "accuracy": float(
        accuracy_score(y_test, test_pred)
    ),
    "sample_count": int(len(y_test))
}

print("Validation metrics:")
print(json.dumps(validation_metrics, indent=4))

print("\nFinal-test metrics:")
print(json.dumps(final_test_metrics, indent=4))


Validation metrics:
{
    "flagged_rate": 0.3,
    "precision": 0.47,
    "recall": 0.7007952286282306,
    "accuracy": 0.7808,
    "sample_count": 5000
}

Final-test metrics:
{
    "flagged_rate": 0.2848,
    "precision": 0.4964887640449438,
    "recall": 0.7027833001988072,
    "accuracy": 0.7968,
    "sample_count": 5000
}


In [16]:
## Creating Json metadata

model_metadata = {
    "version": MODEL_VERSION,
    "trained_on": trained_on,
    "sklearn_version": version("scikit-learn"),

    "features": feature_names,

    "operating_threshold": float(selected_threshold),

    "threshold_policy": {
        "name": "maximum_manual_review_capacity",
        "description": (
            "Select the lowest feasible probability threshold "
            "on validation data such that no more than 30% "
            "of applications are flagged for manual review."
        ),
        "max_flagged_rate": float(MAX_FLAGGED_RATE),
        "selection_dataset": "validation",
        "selection_method": "validation_score_ranking",
        "positive_class": 1,
        "decision_rule": (
            "Flag when predicted default probability "
            "is greater than or equal to operating_threshold."
        )
    },

    "validation_metrics": validation_metrics,
    "final_test_metrics": final_test_metrics,

    "model_type": "LogisticRegression",
    "random_state": RANDOM_STATE,

    "training_notes": (
        "Pipeline refitted on all development rows after "
        "threshold selection. Final test evaluated once. "
        "No retuning performed after final-test evaluation."
    )
}


In [17]:
## Exporting artifacts

joblib.dump(model, MODEL_PATH)

# Save metadata
with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        model_metadata,
        f,
        indent=4
    )

## Shipped artifacts and service metadata verification

The API reads `version`, `features`, and `operating_threshold` from the JSON metadata and verifies that the ordered `features` match the fitted pipeline’s `feature_names_in_`. The JSON also retains the training timestamp, scikit-learn version, threshold policy, and both evaluation summaries.

In [18]:
required_metadata_keys = {
    "version", "trained_on", "sklearn_version", "features",
    "operating_threshold", "threshold_policy",
    "validation_metrics", "final_test_metrics",
}
assert MODEL_PATH.is_file() and METADATA_PATH.is_file()

with METADATA_PATH.open("r", encoding="utf-8") as file:
    exported_metadata = json.load(file)
exported_model = joblib.load(MODEL_PATH)
missing = required_metadata_keys - exported_metadata.keys()
assert not missing, f"Missing metadata keys: {sorted(missing)}"
assert exported_metadata["features"] == exported_model.feature_names_in_.tolist()
assert exported_metadata["operating_threshold"] == float(selected_threshold)
assert 0 < exported_metadata["operating_threshold"] < 1

print("Model artifact:", MODEL_PATH.resolve())
print("Metadata artifact:", METADATA_PATH.resolve())
print("Service-read fields:", {
    key: exported_metadata[key] for key in ("version", "features", "operating_threshold")
})
print("Artifact verification: PASSED")

Model artifact: D:\17.College\02.Course\04.AI_Model_Deployment\01.Assignments\Final_Project\Cerqueira-Daniel-FP\models\loan_default_model_v1.joblib
Metadata artifact: D:\17.College\02.Course\04.AI_Model_Deployment\01.Assignments\Final_Project\Cerqueira-Daniel-FP\models\loan_default_model_v1.json
Service-read fields: {'version': 1, 'features': ['applicant_age', 'employment_status', 'annual_income', 'credit_score', 'loan_amount', 'loan_term_months', 'interest_rate', 'debt_to_income_pct', 'home_ownership', 'previous_defaults'], 'operating_threshold': 0.24523310312331778}
Artifact verification: PASSED
